# 04 — Ticket classifier, end-to-end

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenTracy/OpenTracy/blob/main/notebooks/04_ticket_classifier.ipynb)

This is a **real app**, not a toy. We'll build a support-ticket classifier — the kind of workload a lot of teams already pay GPT-4o for — and walk it through the full OpenTracy loop:

1. **Start naive** — GPT-4o on every ticket. Measure cost.
2. **Add semantic routing** — the router sends easy tickets to a cheaper model. Measure cost again.
3. **Show the wedge** — the next step (distillation) would drop the cost another 10×. We'll link to notebook 05 for that.

By the end you'll have a before/after comparison on a real workload with real numbers.

> **Runtime:** CPU. You need an OpenAI API key. Cost to run the whole notebook: ~$0.05 in API calls (50 tickets × 2 passes).

## Setup

In [2]:
!pip install opentracy -q

In [3]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

## 1. The workload — 50 support tickets

A realistic mix: easy billing questions, technical bug reports, feature requests, a few ambiguous ones.

In [4]:
tickets = [
    # Billing (easy)
    "Where can I download my invoice for March?",
    "My credit card was charged twice, can you refund one?",
    "How do I update my billing address?",
    "I was promised a discount but don't see it on my invoice.",
    "Can I pay in EUR instead of USD?",
    "Cancel my subscription effective today.",
    "I need a receipt for my tax filing.",
    "Why is my plan more expensive this month?",
    "Change my payment method from card to ACH.",
    "Annual vs monthly — which is cheaper?",
    # Technical (hard)
    "Getting 500 errors when calling /v1/users/me — traceback attached, memory spike at 14:02 UTC",
    "Our webhook deliveries have been silently failing for 6 hours, signature mismatch",
    "After upgrading to 2.4.1 the SDK is sending requests with an expired JWT",
    "SSO with Okta broke after the SAML certificate rotation this weekend",
    "Database connections leaking on the analytics worker — PID 8112 holds 1000+ sockets",
    "Rate limit header says 60/min but we're being throttled at 30",
    "CORS suddenly failing in Safari only, Chrome works fine",
    "Race condition in the write path causes duplicate rows under load",
    "GraphQL resolver returning stale cache after invalidation",
    "TLS handshake fails intermittently from our ap-southeast-1 region only",
    # Feature requests (medium)
    "Can we get bulk export to BigQuery?",
    "Please support SAML group-based role mapping",
    "We need a dark mode in the dashboard",
    "Add a Zapier integration",
    "Could you expose the raw SQL for a query in the UI?",
    "Give us per-team API keys with separate quotas",
    "Support for audio input in the chat endpoint would be great",
    "We'd love to filter traces by metadata in the UI",
    "Add webhooks for subscription renewals",
    "Multi-workspace support — switching without logging out",
    # Mixed / ambiguous
    "Is OpenTracy GDPR compliant? Need for procurement review",
    "Hi, I'm evaluating your platform vs LangSmith — key differences?",
    "How do I self-host?",
    "What's your uptime SLA?",
    "Our CFO is asking about pricing for 1M requests/month",
    "Can I try it for free?",
    "Do you have a Terms of Service doc?",
    "Is the data we send to you used for training?",
    "Hello",
    "Thanks!",
    # More billing
    "Please extend my trial for another week",
    "Which plan lets me invite teammates?",
    "I got double charged — please refund",
    # More technical
    "Logs stopped appearing in Datadog after the SDK upgrade",
    "Memory leak in the Python client — see heap dump",
    "HTTPS cert expired on auth.opentracy.ai",
    # More features
    "Exporter to Parquet please",
    "Anthropic Bedrock region config?",
    "Support Azure AD as an auth provider",
    "Python 3.12 support?",
    "Grafana dashboard templates?",
]

print(f"{len(tickets)} tickets loaded")

51 tickets loaded


## 2. The naive baseline — GPT-4o on everything

Classify all 50 tickets with GPT-4o. This is what most teams do on day one.

In [5]:
# ---- OpenTracy platform hook-up (optional but usually what you want) ----
# By default, opentracy.completion() and opentracy.Router go DIRECT to the
# provider (OpenAI, Anthropic, etc.). That means traces, cost accounting,
# routing decisions, and the distillation loop will NOT see your calls.
#
# To make every call pass through the running OpenTracy engine so it
# shows up in the dashboard at http://<host>:3000/ — uncomment the two
# lines below BEFORE the `import opentracy` call in the next cell.
#
# Provider API keys are configured once via the UI (Settings → API Keys)
# or POST /v1/secrets/<provider>; the engine reads them from
# ~/.opentracy/secrets.json inside its container on every call.

# import os
# os.environ["OPENTRACY_ENGINE_URL"] = "http://localhost:8080"  # or your remote host


In [6]:
import opentracy as ot
import time

CATEGORIES = ["billing", "technical", "feature_request", "other"]

PROMPT = (
    "Classify this support ticket into exactly one of: "
    f"{', '.join(CATEGORIES)}. Respond with just the category name, nothing else.\n\n"
    "Ticket: {ticket}"
)

def classify(ticket: str, model: str):
    resp = ot.completion(
        model=model,
        messages=[{"role": "user", "content": PROMPT.format(ticket=ticket)}],
        temperature=0,
        max_tokens=10,
    )
    return {
        "label": resp.choices[0].message.content.strip().lower(),
        "cost": resp._cost,
        "latency_ms": resp._latency_ms,
        "model": model,
    }

print("Classifying with openai/gpt-4o on every ticket...")
t0 = time.time()
baseline = [classify(t, "openai/gpt-4o") for t in tickets]
elapsed = time.time() - t0

baseline_cost = sum(r["cost"] for r in baseline)
baseline_latency = sum(r["latency_ms"] for r in baseline) / len(baseline)

print(f"  total cost:    ${baseline_cost:.5f}")
print(f"  avg latency:   {baseline_latency:.0f}ms")
print(f"  wall time:     {elapsed:.1f}s")

Classifying with openai/gpt-4o on every ticket...
  total cost:    $0.00672
  avg latency:   541ms
  wall time:     30.7s


That's the number to beat. At this rate, 1M tickets/month would cost about:

In [7]:
per_ticket = baseline_cost / len(tickets)
print(f"per-ticket cost:      ${per_ticket:.6f}")
print(f"cost for 1M tickets:  ${per_ticket * 1_000_000:,.0f}/month")

per-ticket cost:      $0.000132
cost for 1M tickets:  $132/month


## 3. Turn on semantic routing

Load the pre-trained router. It'll pick a cheaper model for easy tickets and GPT-4o for the hard ones, all with **zero rule-writing from us**.

In [8]:
router = ot.load_router(cost_weight=1.0, verbose=False)
# cost_weight=1.0 — prefer cheap models, escalate only when the error signal is strong
print(f"router ready, {router.registry.get_model_ids()[:6]}... ({len(router.registry.get_model_ids())} models total)")

router ready, ['gpt-4o-mini', 'gpt-4o', 'ministral-3b-latest', 'mistral-large-latest', 'mistral-small-latest', 'pixtral-12b-2409']... (10 models total)


In [9]:
# See where the router thinks our tickets should go
from collections import Counter

decisions = [router.route(t) for t in tickets]
model_dist = Counter(d.selected_model for d in decisions)

print("Router's plan for these 50 tickets:")
for model, count in model_dist.most_common():
    print(f"  {model:<28}  {count:>2}  ({100*count/len(tickets):.0f}%)")

Router's plan for these 50 tickets:
  gpt-4o                        24  (47%)
  ministral-3b-latest            9  (18%)
  pixtral-12b-2409               8  (16%)
  gpt-3.5-turbo                  4  (8%)
  gpt-4o-mini                    3  (6%)
  ministral-8b-latest            2  (4%)
  gpt-4-turbo                    1  (2%)


Most billing and FAQ tickets → cheap small model. Gnarly technical issues → stay on a strong one. You didn't write a classifier-of-classifiers; the router's 100 pre-trained semantic clusters already capture this.

## 4. Run the routed version

Same tickets, but each one classified by whatever model the router picked.

In [10]:
def classify_routed(ticket: str):
    d = router.route(ticket)
    # Add provider prefix where needed (most OpenAI models need it; mistral ones don't)
    model = d.selected_model
    if model.startswith("gpt") or model.startswith("o1") or model.startswith("o3"):
        model = f"openai/{model}"
    resp = ot.completion(
        model=model,
        messages=[{"role": "user", "content": PROMPT.format(ticket=ticket)}],
        temperature=0,
        max_tokens=10,
    )
    return {
        "label": resp.choices[0].message.content.strip().lower(),
        "cost": resp._cost,
        "latency_ms": resp._latency_ms,
        "model": d.selected_model,
        "cluster": d.cluster_id,
    }

print("Classifying with semantic routing...")
t0 = time.time()
routed = [classify_routed(t) for t in tickets]
elapsed = time.time() - t0

routed_cost = sum(r["cost"] for r in routed)
routed_latency = sum(r["latency_ms"] for r in routed) / len(routed)

print(f"  total cost:   ${routed_cost:.5f}")
print(f"  avg latency:  {routed_latency:.0f}ms")
print(f"  wall time:    {elapsed:.1f}s")

Classifying with semantic routing...


ValueError: No API key for mistral. Set MISTRAL_API_KEY or pass api_key=

## 5. Before / After

In [11]:
def pct_change(old, new):
    return 100 * (old - new) / old

print("=" * 56)
print(f"{'metric':<22} {'baseline':>12} {'routed':>12}   {'Δ':>6}")
print("=" * 56)
print(f"{'total cost (USD)':<22} {baseline_cost:>12.5f} {routed_cost:>12.5f}   -{pct_change(baseline_cost, routed_cost):>4.0f}%")
print(f"{'avg latency (ms)':<22} {baseline_latency:>12.0f} {routed_latency:>12.0f}   -{pct_change(baseline_latency, routed_latency):>4.0f}%")
print()

m_per_month = 1_000_000
baseline_monthly = baseline_cost / len(tickets) * m_per_month
routed_monthly   = routed_cost   / len(tickets) * m_per_month
print(f"Projected cost at 1M tickets/month:")
print(f"  naive GPT-4o:     ${baseline_monthly:>10,.0f}")
print(f"  with routing:     ${routed_monthly:>10,.0f}")
print(f"  savings:          ${baseline_monthly - routed_monthly:>10,.0f}/month")

metric                     baseline       routed        Δ


NameError: name 'routed_cost' is not defined

Accuracy check — did the routed classifier agree with GPT-4o on the labels?

In [12]:
agree = sum(1 for b, r in zip(baseline, routed) if b["label"] == r["label"])
print(f"label agreement: {agree}/{len(tickets)}  ({100*agree/len(tickets):.0f}%)")

disagreements = [(t, b['label'], r['label'], r['model'])
                 for t, b, r in zip(tickets, baseline, routed) if b['label'] != r['label']]
print(f"\ndisagreements ({len(disagreements)}):")
for ticket, b, r, m in disagreements[:5]:
    print(f"  gpt-4o={b:<18} routed={r:<18} ({m})")
    print(f"    '{ticket[:70]}'")

NameError: name 'routed' is not defined

On a small sample there will be a handful of disagreements — usually on ambiguous tickets where both labels are defensible. In production you'd evaluate with a holdout set that has ground-truth labels (see notebook 05 on distillation + evaluation).

## 6. The next step — distillation (the real wedge)

What you just saw is cost reduction from **already-existing** cheap models. The bigger win comes from **distillation**: take all these GPT-4o labels, fine-tune a tiny llama-3.2-1b on them, and the router swaps the tiny student in for the billing/FAQ clusters.

Rough numbers on a classifier workload like this:

| Stage | Model mix | Cost per ticket | Savings |
| --- | --- | --- | --- |
| 1. Naive | GPT-4o everywhere | ~$0.001 | baseline |
| 2. Routing (this notebook) | GPT-4o + gpt-4o-mini + ministral-3b | ~$0.0002 | **80% off** |
| 3. Distilled (notebook 05) | GPT-4o + a custom llama-3.2-1b | ~$0.00002 | **98% off** |

Stage 3 requires a GPU (training) but the student runs on CPU. Walk through it in **notebook 05 — distillation**.

## What you just shipped

- ✅ A working 50-line ticket classifier
- ✅ Real before/after cost numbers on real prompts
- ✅ Agreement measurement between baseline and routed versions
- ✅ A projection of monthly savings at scale

## Next

- **05 — Distillation** → train your own tiny student model from these GPT-4o labels and watch the cost drop another 10×